# Louvain Edge Severance

The GARG-AML pipeline pre-processes every transaction graph with Louvain community detection (resolution = 10, seed = 1997) and keeps only intra-community edges before computing the ego-graph block densities. A reviewer flagged that this step may obscure multi-mule rings that deliberately cross banks / rails. This notebook quantifies **what fraction of edges that pre-processing actually discards**, on every dataset the paper uses.

For each graph we report:

| column | meaning |
|---|---|
| `n_nodes` | number of nodes in the original graph |
| `n_edges_total` | edges before Louvain |
| `n_edges_intra` | edges retained after Louvain (intra-community only) |
| `n_edges_severed` | `n_edges_total - n_edges_intra` |
| `pct_severed` | `100 * n_edges_severed / n_edges_total` |
| `n_communities` | number of Louvain communities |

Both *directed* and *undirected* views of the graph are reported (the undirected one is the headline number, since Louvain operates on the symmetrised graph).

Outputs:

* `results/louvain_edge_severance.csv` &mdash; one row per (dataset, pipeline).
* `results/louvain_edge_severance.pdf` &mdash; boxplot of `pct_severed` by graph type and size.
* `results/preprocessing_severance.csv` &mdash; edges removed and laundering edges among them, per pre-processing setting and per laundering pattern.
* `results/table_severance_patterns_<dataset>.tex` &mdash; that table as a booktabs LaTeX float.


In [ ]:
# Run from the notebook directory; chdir to the repo root so the relative
# paths used elsewhere in the codebase work.
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
print("cwd:", REPO_ROOT)


In [ ]:
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.data.graph_construction import construct_IBM_graph, construct_synthetic_graph
from src.utils.graph_processing import graph_community

OUT_CSV = Path("results/louvain_edge_severance.csv")
OUT_PDF = Path("results/louvain_edge_severance.pdf")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)


## Helper

`edge_severance(G)` runs `graph_community(G)` (the exact function the pipeline uses) and returns the headline counts. Works for `Graph` and `DiGraph`.


In [ ]:
def edge_severance(G):
    """Run Louvain on G via graph_community and report edge-level severance."""
    n_edges_total = G.number_of_edges()
    if n_edges_total == 0:
        return dict(
            n_nodes=G.number_of_nodes(),
            n_edges_total=0, n_edges_intra=0,
            n_edges_severed=0, pct_severed=float("nan"),
            n_communities=0,
        )
    H = graph_community(G)
    n_edges_intra = H.number_of_edges()

    # The number of communities equals the number of weakly-connected
    # components in the reduced graph plus any isolated nodes that ended
    # up alone. graph_community keeps all original nodes, so a cheap
    # estimate is the number of connected components of the undirected
    # view of H.
    H_und = H.to_undirected() if H.is_directed() else H
    n_communities = nx.number_connected_components(H_und)

    return dict(
        n_nodes=G.number_of_nodes(),
        n_edges_total=n_edges_total,
        n_edges_intra=n_edges_intra,
        n_edges_severed=n_edges_total - n_edges_intra,
        pct_severed=100.0 * (n_edges_total - n_edges_intra) / n_edges_total,
        n_communities=n_communities,
    )

def severance_both_views(loader, label):
    """Run severance on the undirected and directed views of the same graph."""
    rows = []
    for directed in [False, True]:
        t0 = time.perf_counter()
        G = loader(directed=directed)
        info = edge_severance(G)
        info.update(
            dataset=label,
            pipeline="directed" if directed else "undirected",
            louvain_seconds=time.perf_counter() - t0,
        )
        rows.append(info)
        del G
    return rows


## IBM datasets

`HI-Small` runs in a couple of minutes. `LI-Large` is ~16 GB on disk; it's gated behind `RUN_LI_LARGE` &mdash; flip the flag if you have time and memory to spare.


In [ ]:
RUN_HI_SMALL = True
RUN_LI_LARGE = False   # ~16 GB CSV, hours of Louvain; opt in explicitly

ibm_rows = []

if RUN_HI_SMALL:
    path = "data/HI-Small_Trans.csv"
    print(f"Loading {path} ...")
    ibm_rows.extend(severance_both_views(
        lambda directed: construct_IBM_graph(path=path, directed=directed),
        label="HI-Small",
    ))

if RUN_LI_LARGE:
    path = "data/LI-Large_Trans.csv"
    print(f"Loading {path} (this will take a while) ...")
    ibm_rows.extend(severance_both_views(
        lambda directed: construct_IBM_graph(path=path, directed=directed),
        label="LI-Large",
    ))

ibm_df = pd.DataFrame(ibm_rows)
ibm_df


## Synthetic datasets

66-point grid: `{Barabasi-Albert, Erdos-Renyi, Watts-Strogatz}` &times; sizes `{100, 10_000, 100_000}` &times; edge/rewiring params &times; `{3, 5}` injected patterns. The graph topology is the same regardless of injection type (`separate` / `new_mules` / `existing_mules`), so each of the 66 edge files is loaded once.

The 100k-node graphs take a few minutes each. `MAX_SYNTHETIC_SIZE` lets you cap the run.


In [ ]:
def construct_dataset_names():
    """Mirror scripts/gargaml_directed_synth.py::construct_datasets() so the
    notebook stays in sync with the experimental grid."""
    names = []
    n_nodes_list = [100, 10000, 100000]
    m_edges_list = [1, 2, 5]
    p_edges_list = [0.001, 0.01]
    generators = ["Barabasi-Albert", "Erdos-Renyi", "Watts-Strogatz"]
    n_patterns_list = [3, 5]

    for n_nodes in n_nodes_list:
        for n_patterns in n_patterns_list:
            if n_patterns > 0.06 * n_nodes:
                continue
            for gen in generators:
                if gen == "Barabasi-Albert":
                    p_edges = 0
                    for m_edges in m_edges_list:
                        names.append(f"synthetic_{gen}_{n_nodes}_{m_edges}_{p_edges}_{n_patterns}")
                elif gen == "Erdos-Renyi":
                    m_edges = 0
                    for p_edges in p_edges_list:
                        names.append(f"synthetic_{gen}_{n_nodes}_{m_edges}_{p_edges}_{n_patterns}")
                else:  # Watts-Strogatz
                    for m_edges in m_edges_list:
                        for p_edges in p_edges_list:
                            names.append(f"synthetic_{gen}_{n_nodes}_{m_edges}_{p_edges}_{n_patterns}")
    return names

DATASET_NAMES = construct_dataset_names()
print(f"{len(DATASET_NAMES)} synthetic datasets in the grid")


In [ ]:
MAX_SYNTHETIC_SIZE = 10_000  # set to 100_000 to include the largest graphs

def parse_size(name):
    return int(name.split("_")[2])

selected = [n for n in DATASET_NAMES if parse_size(n) <= MAX_SYNTHETIC_SIZE]
skipped = [n for n in DATASET_NAMES if parse_size(n) > MAX_SYNTHETIC_SIZE]
print(f"Selected: {len(selected)} | Skipped (size > {MAX_SYNTHETIC_SIZE}): {len(skipped)}")


In [ ]:
synth_rows = []
for name in tqdm(selected):
    path = f"data/edge_data_{name}.csv"
    if not Path(path).is_file():
        print(f"  missing on disk, skipping: {path}")
        continue
    rows = severance_both_views(
        lambda directed, p=path: construct_synthetic_graph(path=p, directed=directed),
        label=name,
    )
    # Tag with grid coordinates for easy slicing in plots / tables.
    parts = name.split("_")
    for row in rows:
        row["generator"] = parts[1]
        row["n_nodes_grid"] = int(parts[2])
        row["m_edges"] = float(parts[3])
        row["p_edges"] = float(parts[4])
        row["n_patterns"] = int(parts[5])
    synth_rows.extend(rows)

synth_df = pd.DataFrame(synth_rows)
print(synth_df.shape)
synth_df.head()


## Combine and persist


In [ ]:
full_df = pd.concat([ibm_df, synth_df], ignore_index=True, sort=False)
full_df.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV} ({len(full_df)} rows)")
full_df.head()


## Headline numbers for the paper

Two summaries:

1. **IBM** &mdash; report exactly the numbers from `ibm_df` in the appendix.
2. **Synthetic** &mdash; mean &plusmn; std of `pct_severed` per (generator, n_nodes), undirected view.


In [ ]:
print("=== IBM ===")
print(ibm_df[["dataset", "pipeline", "n_nodes", "n_edges_total",
              "n_edges_intra", "pct_severed", "n_communities"]].to_string(index=False))


In [ ]:
print("=== Synthetic (undirected view) ===")
agg = (synth_df[synth_df["pipeline"] == "undirected"]
       .groupby(["generator", "n_nodes_grid"])["pct_severed"]
       .agg(["mean", "std", "min", "max", "count"])
       .round(2))
print(agg.to_string())


## Plot


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
undir = synth_df[synth_df["pipeline"] == "undirected"].copy()
undir["size_label"] = undir["n_nodes_grid"].astype(str)
sns.boxplot(
    data=undir, x="size_label", y="pct_severed", hue="generator",
    order=sorted(undir["size_label"].unique(), key=int),
    ax=ax,
)
# Overlay IBM points so the appendix figure shows both regimes.
for _, row in ibm_df[ibm_df["pipeline"] == "undirected"].iterrows():
    ax.scatter([], [], label=f"{row['dataset']}: {row['pct_severed']:.1f}%", marker="*")
ax.set_xlabel("Number of nodes (synthetic)")
ax.set_ylabel("% edges severed by Louvain")
ax.set_title("Louvain edge severance (undirected view, resolution=10, seed=1997)")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.savefig(OUT_PDF)
plt.show()
print(f"wrote {OUT_PDF}")


## Which edges are removed, and are they laundering edges?

The counts above say how much of the graph each pre-processing setting discards. They do not say whether it discards the *interesting* part. This section splits the removed edges by label, for both pre-processing arms of the sensitivity analysis:

* **Louvain** at each resolution in the sweep — inter-community edges are dropped, every account is kept.
* **Hub removal** (`_hubs<k>`) — the `k` highest-degree accounts are deleted along with all their edges, and Louvain is not run at all.

Both use the pipeline's own functions (`community_map`, `graph_degree`), so the arms measured here are exactly the arms scored by `gargaml_directed.py` / `gargaml_undirected.py` under the matching dataset names.

An **edge** is a pair of accounts, with parallel transactions collapsed as the graph collapses them, and is counted as laundering if *any* transaction on the pair is flagged.

The breakdown is reported **per laundering pattern** as well as pooled, since GARG-AML targets `GATHER-SCATTER` / `SCATTER-GATHER` specifically and the pre-processing may treat a one-hop shape (`FAN-OUT`, `FAN-IN`) very differently from a multi-hop one. Pattern membership comes from `pattern_instances`, the same parser the pattern-splitting diagnostic uses, so an edge belongs to a pattern when it carries a transaction of an attempt of that type. Three things to read carefully:

* The `ALL` row is every flagged edge, **not** the sum of the pattern rows: a handful of account pairs appear in attempts of two different types, so the type rows overlap slightly.
* `Not Classified` is the flagged edges belonging to no attempt in the patterns file — on HI-Small that is the largest single bucket, and it is a property of the data, not of the pre-processing.
* `pct_of_removed_laundering` is the laundering share *within* what was removed. Compare it against `pct_laundering_overall` to see whether a setting discards laundering edges disproportionately.

Output: `results/preprocessing_severance.csv` — one row per (dataset, setting, pattern).

In [ ]:
from src.data.bank_views import patterns_path
from src.data.pattern_construction import pattern_instances
from src.utils.graph_processing import community_map, graph_degree

SEVERANCE_CSV = Path("results/preprocessing_severance.csv")

RESOLUTIONS = [1, 5, 10, 20, 50]   # the sweep; 10 is the published setting
HUB_COUNTS = [5, 10, 100]     # the _hubs<k> arms

def undirected_pairs(source, target):
    """(source, target) -> the unordered pair each transaction sits on."""
    a, b = source.astype(str), target.astype(str)
    swap = a > b
    return np.where(swap, b, a), np.where(swap, a, b)

def laundering_edges(path):
    """Simple undirected edges with a laundering flag.

    Mirrors construct_IBM_graph: self-loops dropped, parallel transactions
    collapsed. The flag is 1 when any transaction on the pair is flagged.
    """
    d = pd.read_csv(path, usecols=["Account", "Account.1", "Is Laundering"])
    d = d[d["Account"] != d["Account.1"]]
    u, v = undirected_pairs(d["Account"], d["Account.1"])
    return (pd.DataFrame({"u": u, "v": v, "ml": d["Is Laundering"].values})
              .groupby(["u", "v"], sort=False)["ml"].max().reset_index())

def pattern_masks(edges, path):
    """``{pattern: boolean mask over edges}``, plus ``ALL`` and ``Not Classified``.

    An edge belongs to a pattern when it carries a transaction of an attempt
    of that type. A pair appearing in two types belongs to both, so the type
    masks are not disjoint and only ``ALL`` is a total.
    """
    inst = pattern_instances(path)
    u, v = undirected_pairs(inst["source"], inst["target"])
    inst = pd.DataFrame({"u": u, "v": v, "pattern": inst["pattern_type"].values})
    inst = inst[inst["u"] != inst["v"]].drop_duplicates()

    key = pd.MultiIndex.from_arrays([edges["u"], edges["v"]])
    flagged = edges["ml"].values == 1

    masks = {"ALL": flagged}
    classified = np.zeros(len(edges), dtype=bool)
    for pattern, rows in inst.groupby("pattern"):
        mask = key.isin(pd.MultiIndex.from_arrays([rows["u"], rows["v"]]))
        masks[pattern] = mask
        classified |= mask
    masks["Not Classified"] = flagged & ~classified

    # A pattern edge the graph does not have would be silently invisible here.
    missing = len(inst.drop_duplicates(["u", "v"])) - int(classified.sum())
    if missing:
        print(f"  note: {missing} pattern edges absent from the graph")
    return masks

def removal_row(edges, removed, masks, dataset, setting, pattern):
    """One row: how much was removed, and how much of `pattern` went with it."""
    flagged = masks[pattern]
    n, cut = len(edges), int(removed.sum())
    n_ml, cut_ml = int(flagged.sum()), int((flagged & removed.values).sum())
    return dict(
        dataset=dataset, setting=setting, pattern=pattern,
        edges=n, edges_removed=cut, pct_removed=100.0 * cut / n,
        laundering_edges=n_ml, laundering_removed=cut_ml,
        pct_laundering_removed=100.0 * cut_ml / n_ml if n_ml else float("nan"),
        pct_laundering_overall=100.0 * n_ml / n,
        pct_of_removed_laundering=100.0 * cut_ml / cut if cut else float("nan"),
    )

In [ ]:
sev_rows = []

for label, path in [("HI-Small", "data/HI-Small_Trans.csv")] + (
        [("LI-Large", "data/LI-Large_Trans.csv")] if RUN_LI_LARGE else []):
    print(f"Loading {path} ...")
    edges = laundering_edges(path)
    masks = pattern_masks(edges, patterns_path(label))
    G = construct_IBM_graph(path=path, directed=False)
    print(f"  {len(edges):,} edges, {int(edges['ml'].sum()):,} laundering, "
          f"{len(masks) - 2} pattern types")

    settings = []
    for resolution in RESOLUTIONS:
        communities = community_map(G, resolution)
        settings.append((f"r={resolution}",
                         edges["u"].map(communities) != edges["v"].map(communities)))
        print(f"  r={resolution} done")
    for k in HUB_COUNTS:
        # Ask graph_degree which nodes it deletes rather than re-deriving them,
        # so this matches the graph the measure scripts actually score.
        hubs = set(G) - set(graph_degree(G, n_hubs=k))
        settings.append((f"hubs{k}",
                         edges["u"].isin(hubs) | edges["v"].isin(hubs)))
        print(f"  hubs{k} done")

    for setting, removed in settings:
        for pattern in masks:
            sev_rows.append(removal_row(edges, removed, masks, label, setting, pattern))

sev_df = pd.DataFrame(sev_rows)
sev_df.to_csv(SEVERANCE_CSV, index=False)
print(f"wrote {SEVERANCE_CSV} ({len(sev_df)} rows)")
sev_df.head()

In [ ]:
# Pooled first, then the share of each pattern's edges that each setting removes.
print("=== all flagged edges ===")
print(sev_df[sev_df["pattern"] == "ALL"]
        [["dataset", "setting", "edges_removed", "pct_removed",
          "laundering_removed", "pct_laundering_removed",
          "pct_of_removed_laundering", "pct_laundering_overall"]]
        .round(2).to_string(index=False))

print("\n=== % of each pattern's edges removed ===")
print(sev_df.pivot_table(index="pattern", columns="setting",
                         values="pct_laundering_removed")
        .reindex([p for p in ["GATHER-SCATTER", "SCATTER-GATHER", "FAN-OUT",
                              "FAN-IN", "CYCLE", "BIPARTITE", "STACK", "RANDOM",
                              "Not Classified", "ALL"] if p in set(sev_df["pattern"])])
        .round(2).to_string())

print("\n=== pattern edges in the graph ===")
print(sev_df[sev_df["setting"] == sev_df["setting"].iloc[0]]
        [["pattern", "laundering_edges"]].to_string(index=False))

### LaTeX for the paper

`severance_pattern_latex` renders the table above as a booktabs float (needs `\usepackage{booktabs}`), one file per dataset in `results/`. The percentages are the same numbers printed above; the `All graph edges` row at the bottom is the baseline a pattern row is read against.

In [ ]:
from src.utils.reporting import severance_pattern_latex

for dataset in sev_df["dataset"].unique():
    tex = severance_pattern_latex(sev_df, dataset=dataset,
                                  label=f"tab:severance-patterns-{dataset.lower()}")
    tex_path = Path(f"results/table_severance_patterns_{dataset}.tex")
    tex_path.write_text(tex)
    print(f"wrote {tex_path}\n")
    print(tex)